#The Transformation Logic

In [0]:
query=("""
SELECT
   ROW_NUMBER() OVER (ORDER BY cc.customer_id) AS customer_key,
   cc.customer_id,
   cc.customer_number,
   cc.first_name,
   cc.last_name,
   cl.country,
   cc.marital_status,
   CASE
   WHEN cc.gender != 'n/a' THEN cc.gender
   else COALESCE(ec.gender,'n/a')
   END AS gender,
   ec.birth_date,
   cc.created_date
FROM silver.crm_customers AS cc
LEFT JOIN silver.erp_customers AS ec
   ON cc.customer_number = ec.customer_number
LEFT JOIN silver.erp_customer_location AS cl
   ON cc.customer_number = cl.customer_number
       """)
df=spark.sql(query)


#Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("gold.dim_customers")


#Sanity check for Gold Table

In [0]:
%sql
SELECT * FROM gold.dim_customers
LIMIT 10;